# Tucker rank / 再構成誤差の確認

このNotebookでは、`00_tucker_hosvd_basics.ipynb` で自作したHOSVD/Tucker処理を使い、**multilinear rankを変えたときに圧縮率と再構成誤差がどう変わるか**を確認する。

目的は、CNNへ進む前に「rankを下げるほど小さくなるが、再構成誤差は増える」というTucker圧縮の基本的なtrade-offを確認すること。

このNotebookでは新しい分解アルゴリズムは実装しない。`hosvd` / `reconstruct_tucker` / `relative_frobenius_error` は、前Notebookで完成させた自作関数を使う。


## 1. 実験対象テンソル

rankの違いを比較しやすいよう、固定seedで少し大きめの3階テンソルを作る。


In [ ]:
import torch

torch.manual_seed(0)
X = torch.randn(6, 5, 4)

print("shape:", X.shape)
print("elements:", X.numel())


## 2. 自作関数の準備

`00_tucker_hosvd_basics.ipynb` で完成させた以下の関数を使用する。

- `hosvd`
- `reconstruct_tucker`
- `relative_frobenius_error`

`src` へ共通化する前なので、学習中は完成した実装をこのNotebookでも使える状態にする。


## 3. 比較するrank

まずは全modeを対象にし、rankを段階的に下げて比較する。


In [ ]:
rank_settings = [
    {0: 6, 1: 5, 2: 4},
    {0: 4, 1: 3, 2: 3},
    {0: 3, 1: 2, 2: 2},
    {0: 2, 1: 2, 2: 1},
]

rank_settings


## 4. Tucker表現のパラメータ数

元テンソルの要素数と、`core + factor matrices` が持つ要素数を比較する。


In [ ]:
def tucker_parameter_count(shape, ranks):
    """
    目的: 指定したrankでTucker表現を作ったときの、
          core tensorとfactor matrixの総要素数を求める。

    shape: 元テンソルのshape
    ranks: {mode: rank} 形式の辞書
    """
    pass


部分modeだけを圧縮する場合は、指定していないmodeのdimensionがcoreにそのまま残ることにも注意する。


## 5. rankごとの再構成誤差と圧縮率

各rank設定について、次を記録する。

- rank
- core shape
- Tucker表現のパラメータ数
- 元テンソルに対する圧縮率
- relative Frobenius error


## 6. 結果の確認

rankを下げたときに、次の傾向が出るかを見る。

- coreが小さくなる
- Tucker表現のパラメータ数が減る
- 圧縮率が高くなる
- 再構成誤差は一般に大きくなる

CNNではこの考え方をConv2dの `out channel` / `in channel` rank選択へつなげる。


## 7. partial HOSVDでも確認

CNNのTucker-2を意識して、一部modeだけを圧縮するケースも確認する。

この3階テンソルでは例としてmode 0, 1だけを低rank化し、mode 2はそのまま残す。


In [ ]:
partial_rank_settings = [
    {0: 6, 1: 5},
    {0: 4, 1: 3},
    {0: 3, 1: 2},
    {0: 2, 1: 2},
]

partial_rank_settings


## 8. CNNへ進む前の確認

このNotebookを終えた時点で、次を説明できればCNNのTucker-2へ進む。

1. multilinear rankを下げるとcoreのshapeがどう変わるか
2. factor matrixのパラメータ数がどう決まるか
3. 全mode圧縮とpartial HOSVDの違い
4. 圧縮率と再構成誤差がtrade-offになる理由
5. CNNではなぜ `out channel` / `in channel` のmodeだけを圧縮するのか
